# 05 — Fixed-Target Reliability Summary

Run independently: this notebook loads current completed DB models and champion
results itself. It exports **the fixed-target attainment PNG** plus native and
noise terminal-reliability CSVs. It does not compute adaptive AUC profiles, load
cross-function traces, run evaluations, or display images inline.

Detailed strategy/convergence profiles are in `06_convergence_ecdf_and_ablations.ipynb`;
noise robustness is in `07_noise_robustness.ipynb`. Each export is
cached separately and refreshed only when relevant inputs, settings or code change.


In [1]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "shared").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project directory or notebooks/analysis/.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

# Notebook-owned presentation helpers are imported from the project root.
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import plotly.io as pio
pio.defaults.mathjax = None
from benchmark.infra.module import BenchmarkModule
from shared.config import RESULTS_DIR
from notebooks.analysis.plotting.exporter import AnalysisFigureExporter


figure_exporter = AnalysisFigureExporter(RESULTS_DIR)

# Optional condition filters; None selects the full available matrix.
FILTER_DIMS = None
FILTER_PROBLEMS = None
FILTER_NOISE_STDS = None


analysis_parameters = {
    "budget": 100_000,
    "trials_per_condition": 20,
    "random_seed": 42,
    "first_instance_id": 2,
    "trial_timeout_seconds": 30.0,
    "observation_trace_points": 64,
    "classical_baselines": ["cmaes", "de", "pso"],
    "noise_robustness_enabled": True,
    "cross_function_enabled": False,
    "cross_function_problem_ids": [1, 8, 11, 15, 21],
    "primary_prompt_strategy": "baseline",
    "primary_target": 1e-8,
    "secondary_target": 1e-2,
    "bootstrap_samples": 1000,
    "bootstrap_seed": 20260923,
    "checkpoint_fractions": [0.01, 0.1, 0.5, 1.0],
}

# Initialize this context once; operations manage their own scopes.
import atexit

if "benchmark_module" in globals():
    atexit.unregister(benchmark_module.close)
    benchmark_module.close()
benchmark_module = BenchmarkModule()
atexit.register(benchmark_module.close)


## Calculate reliability (no figure or report export)


In [2]:
reliability = benchmark_module.analyze_reliability(dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS, **analysis_parameters)
reliability_snapshot = benchmark_module.save_analysis_results(reliability)
print(f"Calculated {len(reliability.native)} current native condition rows.")
display(reliability.native)

print(f"Saved numerical results: {reliability_snapshot}")


Calculated 8 current native condition rows.


,Model,Strategy,Solver,Source Problem,Problem ID,Dim,Noise Std,Code Hash,Trials,Complete,Primary Successes,Primary Success Rate,Secondary Success Rate,CI Lower,CI Upper,Median Terminal Error
0,qwen2.5-coder-32b-instruct-q4_k_m.gguf,baseline,qwen_32b_baseline_implicit,15,15,10,0.1,51b39f1166c8a4b2ada5b9508c5ea2ba1310cf3d6680db...,20,True,0,0.0,0.0,0.0,0.161125,17.176272
1,qwen2.5-coder-32b-instruct-q4_k_m.gguf,baseline,qwen_32b_baseline_noisy,15,15,10,0.1,3902ec6f279e4aa01a6cb608b3931ca9139a34342acea6...,20,True,0,0.0,0.0,0.0,0.161125,18.222720
2,qwen2.5-coder-32b-instruct-q4_k_m.gguf,guided,qwen_32b_guided_implicit,15,15,10,0.1,5360e8825f08ccb459bf51573a97438153d74cfb185b66...,20,True,0,0.0,0.0,0.0,0.161125,32.087569
3,qwen2.5-coder-32b-instruct-q4_k_m.gguf,guided,qwen_32b_guided_noisy,15,15,10,0.1,5669b5b786fb8d35eceeaf1929d44051b42679e125cc11...,20,True,0,0.0,0.0,0.0,0.161125,inf
4,qwen2.5-coder-32b-instruct-q4_k_m.gguf,thinking,qwen_32b_thinking_implicit,15,15,10,0.1,72e841ef0e5c71b29cd786a9625fe73f87105c25409ebf...,20,True,0,0.0,0.0,0.0,0.161125,23.643160
5,qwen2.5-coder-32b-instruct-q4_k_m.gguf,thinking,qwen_32b_thinking_noisy,15,15,10,0.1,e289417f70a47330a6de7dc15f8d2407455bcbe9767c75...,20,True,0,0.0,0.0,0.0,0.161125,90.861694
6,qwen2.5-coder-32b-instruct-q4_k_m.gguf,vectorization,qwen_32b_vectorization_implicit,15,15,10,0.1,6758a045e1b33979192b423da873c29e7da0cd3adc41a2...,20,True,0,0.0,0.0,0.0,0.161125,11.589067
7,qwen2.5-coder-32b-instruct-q4_k_m.gguf,vectorization,qwen_32b_vectorization_noisy,15,15,10,0.1,5e4c7f2d3dc546d493f59a916f55f3f8d42020287a83f6...,20,True,0,0.0,0.0,0.0,0.161125,63.443286


Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/reliability/fa4039d609d6f8c411cf644f627d375bab61e9d6120b48941dbbab0c143fec3f


## Reliability tables and fixed-target attainment export

The matrix scores returned solutions. The attainment curve describes the best
clean point queried, which need not be returned. Noise reliability uses identical
frozen champions, not separately synthesized noisy champions. Incomplete conditions
are excluded from aggregate figures, not counted as failures.


In [3]:
figure_exporter.export_reliability(reliability)
